# Dwh NYC Yellow Taxi Data

## Import Packages

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    BooleanType,
    DateType,
    TimestampType,
    DoubleType,
    LongType,
)

In [0]:
from pyspark import pipelines as dp

## Set Variables

## Create Schema

In [0]:
schema = StructType([
    StructField(
        name="VendorID",
        dataType=IntegerType(),
        nullable=True,
        metadata={
            "comment": "Identifier of the technology provider that processed and submitted the trip record"
        }
    ),
    StructField(
        name="tpep_pickup_datetime",
        dataType=TimestampType(),
        nullable=True,
        metadata={
            "comment": "Date and time when the taxi meter was engaged and the trip officially started"
        }
    ),
    StructField(
        name="tpep_dropoff_datetime",
        dataType=TimestampType(),
        nullable=True,
        metadata={
            "comment": "Date and time when the taxi meter was disengaged and the trip ended"
        }
    ),
    StructField(
        name="passenger_count",
        dataType=LongType(),
        nullable=True,
        metadata={
            "comment": "Number of passengers in the vehicle during the trip"
        }
    ),
    StructField(
        name="trip_distance",
        dataType=DoubleType(),
        nullable=True,
        metadata={
            "comment": "Distance traveled during the trip in miles as reported by the taximeter"
        }
    ),
    StructField(
        name="RatecodeID",
        dataType=LongType(),
        nullable=True,
        metadata={
            "comment": "Rate code applied to the trip, including standard, JFK, Newark, negotiated fare, or group ride"
        }
    ),
    StructField(
        name="store_and_fwd_flag",
        dataType=StringType(),
        nullable=True,
        metadata={
            "comment": "Indicates whether the trip record was stored locally before transmission due to communication interruptions"
        }
    ),
    StructField(
        name="PULocationID",
        dataType=IntegerType(),
        nullable=True,
        metadata={
            "comment": "TLC Taxi Zone identifier where the passenger was picked up"
        }
    ),
    StructField(
        name="DOLocationID",
        dataType=IntegerType(),
        nullable=True,
        metadata={
            "comment": "TLC Taxi Zone identifier where the passenger was dropped off"
        }
    ),
    StructField(
        name="payment_type",
        dataType=LongType(),
        nullable=True,
        metadata={
            "comment": "Payment method used by the passenger, such as credit card, cash, no charge, dispute, or unknown"
        }
    ),
    StructField(
        name="fare_amount",
        dataType=DoubleType(),
        nullable=True,
        metadata={
            "comment": "Base fare charged for the trip before taxes, tolls, surcharges, and tips"
        }
    ),
    StructField(
        name="extra",
        dataType=DoubleType(),
        nullable=True,
        metadata={
            "comment": "Additional surcharges applied to the fare, including peak-hour and overnight charges"
        }
    ),
    StructField(
        name="mta_tax",
        dataType=DoubleType(),
        nullable=True,
        metadata={
            "comment": "Mandatory tax collected on behalf of the Metropolitan Transportation Authority"
        }
    ),
    StructField(
        name="tip_amount",
        dataType=DoubleType(),
        nullable=True,
        metadata={
            "comment": "Gratuity amount paid by the passenger"
        }
    ),
    StructField(
        name="tolls_amount",
        dataType=DoubleType(),
        nullable=True,
        metadata={
            "comment": "Total toll charges incurred during the trip"
        }
    ),
    StructField(
        name="improvement_surcharge",
        dataType=DoubleType(),
        nullable=True,
        metadata={
            "comment": "Regulatory surcharge supporting taxi industry improvement initiatives"
        }
    ),
    StructField(
        name="total_amount",
        dataType=DoubleType(),
        nullable=True,
        metadata={
            "comment": "Total amount paid by the passenger including fare, taxes, tolls, surcharges, and tip"
        }
    ),
    StructField(
        name="congestion_surcharge",
        dataType=DoubleType(),
        nullable=True,
        metadata={
            "comment": "Congestion surcharge applied to eligible trips operating in designated congestion zones"
        }
    ),
    StructField(
        name="Airport_fee",
        dataType=DoubleType(),
        nullable=True,
        metadata={
            "comment": "Airport access fee applied to eligible airport-related trips"
        }
    ),
    StructField(
        name="cbd_congestion_fee",
        dataType=DoubleType(),
        nullable=True,
        metadata={
            "comment": "Central Business District congestion pricing fee applied to qualifying trips"
        }
    ),
    StructField(
        name="sys_Insert_Dt",
        dataType=TimestampType(),
        nullable=False,
        metadata={
            "comment": "Timestamp when the record was ingested into the platform"
        }
    ),
    StructField(
        name="sys_Insert_Fp",
        dataType=StringType(),
        nullable=False,
        metadata={
            "comment": "Source file path or ingestion source identifier used to load the record"
        }
    )
])

## ETL

In [0]:
@dp.temporary_view(name="dwh_nyc_taxi_basis")
def bronze_dwh_taxi_basis():
    df = spark.read.parquet("/Volumes/analytics/route_analysis/yellowcab_info/*.parquet")
    df = df.withColumn("sys_Insert_Dt", F.current_timestamp())
    df = df.withColumn("sys_Insert_Fp", F.col("_metadata.file_path"))
    
    # Drop duplicates
    df = df.dropDuplicates(["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID","VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID"])

    # Fit Dataframe to schema
    schema_columns = [(field.name, field.dataType) for field in schema.fields]
    df = df.select([F.col(col_name).cast(col_dtype) for col_name, col_dtype in schema_columns])
    return df

dp.create_streaming_table("analytics.bronze.dwh_nyc_taxi", comment="This table shows the trip Information of 2025 for the NYC Taxis", schema=schema)

dp.create_auto_cdc_from_snapshot_flow(
    target="analytics.bronze.dwh_nyc_taxi",
    source="dwh_nyc_taxi_basis",
    keys=["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID"],
    stored_as_scd_type=1,
)